# 第 2 周额外周末练习 —— 技术问答原型

## 练习目标（理念）

把第 1 周的「技术问题 / 回答器」升级成可用原型，综合运用第 2 周知识点：

- **Gradio UI**：聊天界面，而不是只在笔记本里 `print`
- **流式（streaming）**：边生成边显示
- **System Prompt**：注入「资深技术研究员」等专业人设
- **多模型切换**：经 OpenRouter 在 Claude / GPT / Gemini 间切换
- **奖励**：工具调用（`search_tech_docs`）+ SQLite 对话日志

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| OpenAI 兼容客户端 | `base_url` 指向 OpenRouter |
| Tool / Function Calling | `tools` + `tool_calls` 回路 |
| Streaming | 最终回复 `stream=True` + `yield` |
| Gradio Tabs / ChatInterface | AI Researcher + Database Logs |

## 怎么跑

1. `.env` 里配置 `OPENROUTER_API_KEY`
2. 自上而下运行代码格（先 `init_db`，再定义工具与 `agent_chat`）
3. 最后一格 `demo.launch(...)` 打开 Gradio

可选挑战：音频输入 / TTS 回复（本笔记本未实现，可作为扩展）。


In [ ]:
# ========== 导入：OpenRouter 客户端 + Gradio + SQLite 日志所需 ==========

# 导入标准库 os：读环境变量（如 OPENROUTER_API_KEY）
import os
# 导入标准库 sqlite3：本地文件型数据库，记每次问答
import sqlite3
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：OpenRouter 提供 OpenAI 兼容 API，可复用此 SDK
from openai import OpenAI
# 导入 gradio：快速搭 Web 聊天 UI
import gradio as gr
# 从 datetime 导入 datetime：给日志行打时间戳
from datetime import datetime


In [ ]:
# ========== 客户端：经 OpenRouter 统一访问多家模型 ==========

# 加载 .env（默认覆盖行为保持原库默认，不额外传 override）
load_dotenv()
# 如果不使用 .env，请在此处手动设置：
# os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-your-key-here"

# base_url 指向 OpenRouter；api_key 从环境变量读取（URL / 变量名勿改）
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)


In [ ]:
# ========== SQLite：建表 + 写入交互日志 ==========

def init_db():
    # 连接（或创建）本地库文件 research_agent.db
    conn = sqlite3.connect("research_agent.db")
    # 取游标，用来执行 SQL
    cursor = conn.cursor()
    # 若 logs 表不存在则创建：时间戳 / 模型 / 提问 / 回答
    cursor.execute('''CREATE TABLE IF NOT EXISTS logs 
                      (timestamp TEXT, model TEXT, prompt TEXT, response TEXT)''')
    # 提交 DDL
    conn.commit()
    # 关掉连接，避免文件锁长期占用
    conn.close()

def log_interaction(model, prompt, response):
    # 每次对话结束后插入一行
    conn = sqlite3.connect("research_agent.db")
    cursor = conn.cursor()
    # 四元组顺序必须与建表列一致；时间格式化成可读字符串
    cursor.execute("INSERT INTO logs VALUES (?, ?, ?, ?)", 
                   (datetime.now().strftime("%Y-%m-%d %H:%M:%S"), model, prompt, response))
    conn.commit()
    conn.close()

# 笔记本启动时先确保表存在
init_db()
# 给人看的就绪提示（不影响逻辑）
print("✅ Database and Client Initialized.")


In [ ]:
# ========== 工具：模拟「技术手册」检索 + OpenAI tools schema ==========

# 导入 json：后面解析 tool_call.function.arguments（JSON 字符串）
import json

# 模拟技术数据库搜索：真正项目里可换成向量库 / HTTP API
def search_tech_docs(query):
    """在内存字典里按关键字检索技术说明（演示用 Tool）。"""
    # 极简知识库：key 是主题词，value 是返回给模型的观察结果
    docs = {
        "api": "API stands for Application Programming Interface. It allows software to talk.",
        "gradio": "Gradio is an open-source Python library used to build ML web apps.",
        "sqlite": "SQLite is a C-language library that implements a small, fast SQL database engine."
    }
    # 如果找到关键字则返回文档，否则返回默认消息
    for key in docs:
        # 不区分大小写：query 里包含 key 就算命中
        if key in query.lower():
            return docs[key]
    # 未命中时的英文回落文案（影响模型后续推理，保持原样）
    return "No specific documentation found for this topic."

# 定义 OpenAI/OpenRouter 格式的工具清单，交给 chat.completions 的 tools=
tools = [
    {
        "type": "function",
        "function": {
            # 名称必须与上面 Python 函数名一致，模型才会对得上
            "name": "search_tech_docs",
            # description 给模型看：何时该调用此工具（英文勿改）
            "description": "Search the internal technical manual for coding definitions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The technical topic to look up."}
                },
                "required": ["query"],
            },
        },
    }
]


In [ ]:
# ========== （重复格）工具定义再写一遍：与上一格逻辑相同，便于单独重跑 ==========

# 再次导入 json（本格可独立执行时需要）
import json

# 模拟技术数据库搜索（与上一格同名函数，后定义会覆盖前定义）
def search_tech_docs(query):
    """在内存字典里按关键字检索技术说明（演示用 Tool）。"""
    docs = {
        "api": "API stands for Application Programming Interface. It allows software to talk.",
        "gradio": "Gradio is an open-source Python library used to build ML web apps.",
        "sqlite": "SQLite is a C-language library that implements a small, fast SQL database engine."
    }
    # 如果找到关键字则返回文档，否则返回默认消息
    for key in docs:
        if key in query.lower():
            return docs[key]
    return "No specific documentation found for this topic."

# 定义 OpenAI/OpenRouter 格式的工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tech_docs",
            "description": "Search the internal technical manual for coding definitions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The technical topic to look up."}
                },
                "required": ["query"],
            },
        },
    }
]


In [ ]:
# ========== agent_chat：Tool 回路 + 流式最终回答 + 写日志 ==========

def agent_chat(message, history, model_name, system_prompt):
    # UI 显示名 → OpenRouter 模型 id（字符串必须与路由可用模型一致）
    models = {
        "Claude 3.5 Sonnet": "anthropic/claude-3.5-sonnet",
        "GPT-4o": "openai/gpt-4o",
        "Gemini 1.5 Pro": "google/gemini-pro-1.5"
    }
    
    # 先放 system，再回放 Gradio history（tuples：user, assistant）
    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    # 追加本轮用户消息
    messages.append({"role": "user", "content": message})

    # 第 1 步：首次调用以查看是否需要工具（非流式，便于读 tool_calls）
    response = client.chat.completions.create(
        model=models[model_name],
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    # 取出助手消息对象（可能带 tool_calls）
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # 步骤 2：处理工具调用（如果存在）
    if tool_calls:
        for tool_call in tool_calls:
            # arguments 是 JSON 字符串，要 loads 成 dict
            function_args = json.loads(tool_call.function.arguments)
            # 本地执行 search_tech_docs，得到 observation
            observation = search_tech_docs(function_args.get("query"))
            
            # 把「助手要调工具」那条消息放回对话
            messages.append(response_message)
            # 再追加 role=tool 的观察结果，供第二轮生成最终答案
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "search_tech_docs",
                "content": observation,
            })
    
    # 第 3 步：传输最终响应（流式；Gradio 需要累计字符串再 yield）
    final_stream = client.chat.completions.create(
        model=models[model_name],
        messages=messages,
        stream=True
    )

    full_text = ""
    for chunk in final_stream:
        # 有的 chunk 只有 role / 空 delta，要判空
        if chunk.choices[0].delta.content:
            full_text += chunk.choices[0].delta.content
            yield full_text
            
    # 第 4 步：登录 SQLite（流结束后用完整文本落库）
    log_interaction(model_name, message, full_text)


In [ ]:
# ========== Gradio UI：聊天 Tab + 数据库日志 Tab ==========

# 导入 pandas：把 SQLite 查询结果展示成表格
import pandas as pd
# 再次导入 sqlite3 / gradio，便于本格单独重跑
import sqlite3
import gradio as gr

# 从 SQLite 为 Gradio UI 提取数据的函数
def get_history():
    try:
        # 读 logs 表，按时间倒序
        conn = sqlite3.connect("research_agent.db")
        df = pd.read_sql_query("SELECT * FROM logs ORDER BY timestamp DESC", conn)
        conn.close()
        return df
    except Exception as e:
        # 库还不存在或查询失败时，返回提示型 DataFrame（文案原样）
        return pd.DataFrame({"Status": ["No history found yet. Start a chat!"]})

# --- Gradio UI 布局（第 5 天：块和选项卡）---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    # 标题与副标题（界面字符串保持原样）
    gr.Markdown("#  Research & Logging Agent")
    gr.Markdown("### Week 2 Exercise")
    
    with gr.Tabs():
        # TAB 1：人工智能研究员
        with gr.TabItem("AI Researcher"):
            with gr.Row():
                # 下拉选「Agent Brain」：选项必须与 agent_chat 里 models 字典 key 一致
                model_sel = gr.Dropdown(
                    choices=["Claude 3.5 Sonnet", "GPT-4o", "Gemini 1.5 Pro"], 
                    value="Claude 3.5 Sonnet", 
                    label="Agent Brain"
                )
                # 可编辑 System Prompt；默认英文人设字符串勿改语义
                sys_prompt = gr.Textbox(
                    value="You are a Senior Technical Researcher. Use your tools to provide accurate data.", 
                    label="System Instructions"
                )
            
            # 流媒体聊天界面：additional_inputs 把模型与 system 传进 agent_chat
            gr.ChatInterface(
                fn=agent_chat, 
                additional_inputs=[model_sel, sys_prompt], 
                type="tuples"
            )
            
        # TAB 2：数据库日志（固定缩进）
        with gr.TabItem("Database Logs"):
            gr.Markdown("### 📜 Conversation History")
            gr.Markdown("Data below is pulled in real-time from `research_agent.db`.")
            
            # 手动刷新按钮
            refresh_btn = gr.Button("🔄 Refresh History", variant="primary")
            
            # 将 SQLite 数据显示为干净的表；value=get_history 表示初始调用该函数
            history_table = gr.DataFrame(value=get_history)
            
            # 将按钮链接到刷新功能
            refresh_btn.click(fn=get_history, outputs=history_table)

# 启动应用程序：share=True 会尝试生成公网链接；debug=True 打印更多错误
demo.launch(share=True, debug=True)
